# 🌍 Global Migration Observatory — Phase 1: Data Exploration

**Objective**: Explore, inspect, and validate the UN DESA International Migrant Stock (2020 Revision) and World Bank World Development Indicators datasets using the modular data pipeline in `src/`.

> **Important Methodological Distinction**:
> **Migrant Stock $\neq$ Annual Migration Flow**.
> Migrant stock represents the estimated total number of foreign-born / foreign citizens residing in a country at mid-year.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
import plotly.express as px
from src.data_loader import load_processed_data, load_raw_migration_destination, load_world_bank_data
from src.data_cleaning import clean_migration_destination, clean_world_bank_data
from src.data_merging import merge_country_level
from src.feature_engineering import compute_migrant_stock_pct_population, compute_period_stock_change
from src.data_validation import DataValidator

pd.set_option('display.max_columns', 20)
print('Pipeline modules loaded successfully.')

## 1. Load & Inspect Processed Destination Migrant Stock

In [ ]:
df_dest = load_processed_data('migration_destination_cleaned.csv')
print(f'Destination Migrant Stock Shape: {df_dest.shape}')
print(f'Unique Countries/Areas: {df_dest["country_code"].nunique()}')
print(f'Years Covered: {sorted(df_dest["year"].unique())}')
df_dest.head(10)

## 2. Load & Inspect Processed World Bank Socioeconomic Indicators

In [ ]:
df_wb = load_processed_data('world_bank_cleaned.csv')
print(f'World Bank Dataset Shape: {df_wb.shape}')
print(f'Unique Countries/Aggregates: {df_wb["country_code"].nunique()}')
print(f'Years Covered: {df_wb["year"].min()} - {df_wb["year"].max()}')
df_wb.head(10)

## 3. Inspect Merged Country Socioeconomic Dataset & Derived Metrics

In [ ]:
df_merged = load_processed_data('migration_country_socioeconomic.csv')
print(f'Merged Dataset Shape: {df_merged.shape}')
overlap = df_merged[df_merged['migrant_stock'].notna() & df_merged['population'].notna()].shape[0]
print(f'Non-null Migrant Stock & Population Overlap: {overlap}')
df_merged[df_merged['year'] == 2020].sort_values('migrant_stock', ascending=False).head(10)[[
    'country', 'country_code', 'year', 'migrant_stock', 'population', 'migrant_stock_pct_population', 'gdp_per_capita'
]]

## 4. Exploratory Visualizations
Visualizing the top sovereign destination countries by migrant stock and stock percentage of population.

In [ ]:
top_2020 = df_merged[(df_merged['year'] == 2020) & (~df_merged['is_aggregate'])].sort_values('migrant_stock', ascending=False).head(15)

fig1 = px.bar(
    top_2020,
    x='country',
    y='migrant_stock',
    title='Top 15 Destination Countries by International Migrant Stock (2020)',
    labels={'country': 'Destination Country', 'migrant_stock': 'Migrant Stock (Foreign-Born)'},
    color='migrant_stock_pct_population',
    color_continuous_scale='Blues'
)
fig1.show()

In [ ]:
# Global Migrant Stock Trends over 1990-2020 for Selected Major Destinations
selected_countries = ['USA', 'DEU', 'GBR', 'FRA', 'CAN', 'AUS', 'ESP', 'ITA']
trend_df = df_merged[df_merged['country_code'].isin(selected_countries)]

fig2 = px.line(
    trend_df,
    x='year',
    y='migrant_stock',
    color='country',
    markers=True,
    title='Migrant Stock Trajectories (1990-2020) Across Major Destinations',
    labels={'year': 'Census Round Year', 'migrant_stock': 'Migrant Stock'}
)
fig2.show()

## 5. Summary & Verification
The ingestion, cleaning, country standardization (ISO-3), entity classification, and validation pipelines executed with 100% test pass rate and clean separation between stocks and flows.